# Notebook 04: Feature Engineering and Preprocessing

**Project:** Predicting NFL Player Value to Identify Contract Overpayment Risk Using Machine Learning

## Objective

The purpose of this notebook is to transform the cleaned wide receiver dataset into a modeling ready dataset for predicting next season receiving yards.

This notebook creates the target variable, engineers football relevant features, evaluates missing values, removes non predictive variables, and prepares the final dataset used for machine learning model development.

Special attention is given to preventing data leakage by ensuring that only information available before the prediction season is used as model input.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

In [2]:
data_path = Path("../data/processed/wr_master.csv")

wr_master = pl.read_csv(data_path)

# Verify successful import
print("Dataset loaded successfully.")
print(f"Rows: {wr_master.height:,}")
print(f"Columns: {wr_master.width:,}")

Dataset loaded successfully.
Rows: 3,148
Columns: 181


## Create the Prediction Target

### Purpose

Each observation represents a player's performance during season **t**. The target is that same player's **receiving yards during season t + 1**.

This prevents incorrectly linking seasons separated by career gaps and avoids introducing invalid training observations.

In [3]:
wr_sorted = (
    wr_master
    .sort(["player_id", "season"])
)

wr_sorted.select(
    [
        "player_name",
        "season",
        "receiving_yards"
    ]
).head(10)

player_name,season,receiving_yards
str,i64,i64
"""D.Driver""",2012,77
"""R.Moss""",2012,434
"""B.Stokley""",2012,544
"""B.Stokley""",2013,115
"""P.Burress""",2012,42
"""S.Smith""",2012,1174
"""S.Smith""",2013,745
"""S.Smith""",2014,1065
"""S.Smith""",2015,670



### Create the Next-Season Receiving-Yards Target

The next recorded season and receiving-yard total are identified within each player's career using a grouped lead operation.

The target is assigned only when:

\[
\text{next season} = \text{current season} + 1
\]

If the next recorded season is not consecutive, the target remains missing. This prevents seasons separated by gaps from being treated as valid one-year forecasting observations.

In [4]:
# Identify the next recorded season and receiving-yard total
wr_target = (
    wr_sorted
    .with_columns(
        [
            pl.col("season")
            .shift(-1)
            .over("player_id")
            .alias("next_recorded_season"),

            pl.col("receiving_yards")
            .shift(-1)
            .over("player_id")
            .alias("next_recorded_receiving_yards")
        ]
    )
    .with_columns(
        pl.when(
            pl.col("next_recorded_season")
            == pl.col("season") + 1
        )
        .then(pl.col("next_recorded_receiving_yards"))
        .otherwise(None)
        .alias("next_season_receiving_yards")
    )
)

wr_target.select(
    [
        "player_name",
        "season",
        "receiving_yards",
        "next_recorded_season",
        "next_recorded_receiving_yards",
        "next_season_receiving_yards"
    ]
).head(20)

player_name,season,receiving_yards,next_recorded_season,next_recorded_receiving_yards,next_season_receiving_yards
str,i64,i64,i64,i64,i64
"""D.Driver""",2012,77,null,null,null
"""R.Moss""",2012,434,null,null,null
"""B.Stokley""",2012,544,2013,115,115
"""B.Stokley""",2013,115,null,null,null
"""P.Burress""",2012,42,null,null,null
…,…,…,…,…,…
"""R.Wayne""",2014,779,null,null,null
"""D.Stallworth""",2012,63,null,null,null
"""J.Gaffney""",2012,68,null,null,null


## Validate Consecutive Season Matching

### Purpose

Before using the target for machine learning, the consecutive season rule is verified.

The validation confirms that:

- Consecutive seasons receive a valid prediction target.
- Career gaps do not produce incorrect targets.
- Final career seasons remain missing because no future season exists.

This quality control step provides evidence that the target creation logic behaves as intended.

In [5]:
# Find observations where the next recorded season is NOT consecutive
season_gaps = (
    wr_target
    .filter(
        (pl.col("next_recorded_season").is_not_null()) &
        (pl.col("next_recorded_season") != pl.col("season") + 1)
    )
    .select(
        [
            "player_name",
            "season",
            "next_recorded_season",
            "next_season_receiving_yards"
        ]
    )
)

print(f"Rows with season gaps: {season_gaps.height}")

season_gaps.head(20)

Rows with season gaps: 115


player_name,season,next_recorded_season,next_season_receiving_yards
str,i64,i64,i64
"""B.Lloyd""",2012,2014,null
"""C.Roby""",2012,2014,null
"""D.Hagan""",2012,2014,null
"""J.Nelson""",2014,2016,null
"""J.Simpson""",2013,2015,null
…,…,…,…
"""J.Morgan""",2012,2014,null
"""C.Matthews""",2015,2017,null
"""T.Pryor""",2013,2015,null


### Validation Summary

The validation confirmed that the target creation logic behaves as intended.

- Consecutive player seasons were successfully assigned a valid `next_season_receiving_yards` target.
- Observations containing season gaps (115 total) were intentionally excluded from target assignment by assigning a null target value.
- Final career seasons also retain null target values because no subsequent season exists.

This validation ensures that the prediction target represents only true one season forecasting observations and prevents invalid player season pairings from entering the machine learning pipeline.

## Feature Inventory and Selection Strategy

The processed dataset contains 182 variables following creation of the prediction target. Rather than including every available variable in the predictive model, each variable is evaluated according to its contribution to forecasting next-season receiving yards.

Variables are classified into one of three categories:

- **Target** – the response variable to be predicted.
- **Keep** – variables representing information available before the prediction season that may contribute predictive value.
- **Remove** – identifiers, administrative fields, redundant metadata, and variables that provide little or no predictive information.

This approach improves model interpretability, supports reproducibility, and helps prevent data leakage by ensuring that only appropriate information is available to the predictive model.

In [6]:
# Create a feature inventory

feature_inventory = (
    pl.DataFrame(
        {
            "feature": wr_target.columns,
            "data_type": [str(dtype) for dtype in wr_target.dtypes]
        }
    )
)

print(f"Total Features: {feature_inventory.height}")

feature_inventory

Total Features: 184


feature,data_type
str,str
"""player_id""","""String"""
"""player_name""","""String"""
"""player_display_name""","""String"""
"""position""","""String"""
"""position_group""","""String"""
…,…
"""draft_pick""","""Int64"""
"""draft_team""","""String"""
"""next_recorded_season""","""Int64"""


In [7]:
# Remove temporary helper columns used during target creation

wr_model = (
    wr_target.drop(
        [
            "next_recorded_season",
            "next_recorded_receiving_yards"
        ]
    )
)

print(f"Rows: {wr_model.height:,}")
print(f"Columns: {wr_model.width:,}")

Rows: 3,148
Columns: 182


## Feature Engineering

Raw football statistics measure volume but don't fully capture player efficiency or performance quality.

Engineered variables provide the predictive model with more informative inputs while remaining interpretable from both football and business perspectives.

In [8]:
# Engineer receiving efficiency features

wr_model = (
    wr_model
    .with_columns(
        [
            (
                pl.when(pl.col("targets") > 0)
                .then(pl.col("receptions") / pl.col("targets"))
                .otherwise(None)
            ).alias("catch_rate"),

            (
                pl.when(pl.col("receptions") > 0)
                .then(pl.col("receiving_yards") / pl.col("receptions"))
                .otherwise(None)
            ).alias("yards_per_reception"),

            (
                pl.when(pl.col("targets") > 0)
                .then(pl.col("receiving_yards") / pl.col("targets"))
                .otherwise(None)
            ).alias("yards_per_target")
        ]
    )
)

wr_model.select(
    [
        "player_name",
        "season",
        "targets",
        "receptions",
        "receiving_yards",
        "catch_rate",
        "yards_per_reception",
        "yards_per_target"
    ]
).head()

player_name,season,targets,receptions,receiving_yards,catch_rate,yards_per_reception,yards_per_target
str,i64,i64,i64,i64,f64,f64,f64
"""D.Driver""",2012,13,8,77,0.615385,9.625,5.923077
"""R.Moss""",2012,50,28,434,0.56,15.5,8.68
"""B.Stokley""",2012,59,45,544,0.762712,12.088889,9.220339
"""B.Stokley""",2013,21,13,115,0.619048,8.846154,5.47619
"""P.Burress""",2012,7,3,42,0.428571,14.0,6.0


### Scoring and Chain-Moving Efficiency

Receiving production is not measured only by total yardage. Productive receivers consistently convert receptions into touchdowns and first downs.

These features quantify a player's ability to create scoring opportunities and sustain offensive drives.

In [9]:
# Engineer scoring and chain-moving efficiency features

wr_model = (
    wr_model
    .with_columns(
        [
            (
                pl.when(pl.col("receptions") > 0)
                .then(pl.col("receiving_tds") / pl.col("receptions"))
                .otherwise(None)
            ).alias("tds_per_reception"),

            (
                pl.when(pl.col("receptions") > 0)
                .then(pl.col("receiving_first_downs") / pl.col("receptions"))
                .otherwise(None)
            ).alias("first_downs_per_reception")
        ]
    )
)

wr_model.select(
    [
        "player_name",
        "season",
        "receptions",
        "receiving_tds",
        "receiving_first_downs",
        "tds_per_reception",
        "first_downs_per_reception"
    ]
).head()

player_name,season,receptions,receiving_tds,receiving_first_downs,tds_per_reception,first_downs_per_reception
str,i64,i64,i64,i64,f64,f64
"""D.Driver""",2012,8,2,5,0.25,0.625
"""R.Moss""",2012,28,3,22,0.107143,0.785714
"""B.Stokley""",2012,45,5,30,0.111111,0.666667
"""B.Stokley""",2013,13,0,8,0.0,0.615385
"""P.Burress""",2012,3,1,3,0.333333,1.0


## Data Preprocessing

The preprocessing stage removes variables that do not contribute predictive information, excludes observations without a valid prediction target, and verifies the quality of the remaining data.

In [10]:
# Remove identifier and metadata columns

columns_to_remove = [
    "player_name",
    "player_display_name",
    "position",
    "position_group",
    "headshot_url",
    "gsis_id",
    "espn_id",
    "yahoo_id",
    "rotowire_id",
    "pff_id",
    "sleeper_id"
]

columns_to_remove = [
    col for col in columns_to_remove
    if col in wr_model.columns
]

wr_model = wr_model.drop(columns_to_remove)

print(f"Removed {len(columns_to_remove)} identifier columns.")
print(f"Remaining columns: {wr_model.width}")

Removed 7 identifier columns.
Remaining columns: 180


In [11]:
# Keep only observations with a valid prediction target

wr_model = (
    wr_model.filter(
        pl.col("next_season_receiving_yards").is_not_null()
    )
)

print(f"Rows available for modeling: {wr_model.height:,}")
print(f"Columns: {wr_model.width}")

Rows available for modeling: 2,127
Columns: 180


## Data Quality Assessment

The assessment verifies:

- Missing values
- Duplicate observations
- Data types

These checks ensure the dataset is suitable for machine learning and improve the reproducibility of the analytical workflow.

In [12]:
# Count missing values by feature

missing_summary = (
    wr_model.null_count()
    .transpose(
        include_header=True,
        header_name="feature",
        column_names=["missing_values"]
    )
    .filter(pl.col("missing_values") > 0)
    .sort("missing_values", descending=True)
)

missing_summary

feature,missing_values
str,u32
"""fg_long""",2127
"""fg_pct""",2127
"""fg_made_list""",2127
"""fg_missed_list""",2127
"""fg_blocked_list""",2127
…,…
"""nfl_id""",22
"""pff_position""",20
"""otc_id""",15


### Future-Information Leakage Control

A feature audit identified two player metadata variables, last_season and
years_of_experience, that are not appropriate for historical next season
prediction.

These variables originate from static player metadata and are joined to
season level observations by player identifier. As a result, their values can
contain career information that would not have been available at the time of
an earlier historical prediction.

Both variables are therefore excluded from the modeling feature set to ensure
that predictors reflect only information that could reasonably have been
available at the prediction season.

Player experience remains conceptually relevant, but a future version should
derive experience directly from season specific historical information rather
than use the static metadata field.

In [14]:
# Remove non-predictive, redundant, and future-information features

additional_columns_to_remove = [
    "fg_long",
    "fg_pct",
    "fg_made_list",
    "fg_missed_list",
    "fg_blocked_list",
    "headshot",
    "nfl_id",
    "otc_id",
    "pfr_id",
    "pfr_position",

    # Removed after feature audit
    "fantasy_points",
    "fantasy_points_ppr",

    # Static player metadata that contains information
    # unavailable at the historical prediction season
    "last_season",
    "years_of_experience",
]

columns_to_remove = [
    col for col in additional_columns_to_remove
    if col in wr_model.columns
]

wr_model = wr_model.drop(columns_to_remove)

print("Additional columns removed:")
for col in columns_to_remove:
    print(f"  - {col}")

print(f"\nRemaining columns: {wr_model.shape[1]:,}")

Additional columns removed:
  - fg_long
  - fg_pct
  - fg_made_list
  - fg_missed_list
  - fg_blocked_list
  - headshot
  - nfl_id
  - otc_id
  - pfr_id
  - fantasy_points
  - fantasy_points_ppr
  - last_season
  - years_of_experience

Remaining columns: 167


In [15]:
print("player_id" in wr_model.columns)

True


In [16]:
# Check for duplicate observations

duplicate_rows = (
    wr_model.height -
    wr_model.unique().height
)

print(f"Duplicate rows: {duplicate_rows}")

Duplicate rows: 0


In [17]:
# Review data types

dtype_summary = (
    pl.DataFrame(
        {
            "feature": wr_model.columns,
            "data_type": [str(dt) for dt in wr_model.dtypes]
        }
    )
)

dtype_summary.group_by("data_type").len().sort("data_type")

data_type,len
str,u32
"""Float64""",16
"""Int64""",122
"""String""",29


In [18]:
# Final removal of non-predictive string features

final_columns_to_remove = [
    "birth_date",
    "college_conference",
    "college_name",
    "common_first_name",
    "display_name",
    "draft_team",
    "esb_id",
    "first_name",
    "football_name",
    "gwfg_distance_list",
    "last_name",
    "latest_team",
    "ngs_position",
    "ngs_position_group",
    "ngs_status",
    "ngs_status_short_description",
    "pat_pct",
    "pff_position",
    "pff_status",
    "position_group_right",
    "position_right",
    "pt_long",
    "recent_team",
    "short_name",
    "smart_id",
    "status",
    "suffix"
]

if (
    "season_type" in wr_model.columns
    and wr_model["season_type"].n_unique() == 1
):
    final_columns_to_remove.append("season_type")

final_columns_to_remove = [
    column
    for column in final_columns_to_remove
    if column in wr_model.columns
]

wr_model = wr_model.drop(final_columns_to_remove)

print(f"Rows: {wr_model.height:,}")
print(f"Final modeling columns: {wr_model.width}")

Rows: 2,127
Final modeling columns: 139


In [19]:
output_path = Path("../data/processed/wr_model.csv")

wr_model.write_csv(output_path)

print(f"Dataset saved to: {output_path}")
print(f"Rows saved: {wr_model.height:,}")
print(f"Columns saved: {wr_model.width}")

Dataset saved to: ..\data\processed\wr_model.csv
Rows saved: 2,127
Columns saved: 139


## Final Dataset Summary

### Final Modeling Dataset

| Metric | Value |
|---|---:|
| Player-season observations | 2,127 |
| Predictor variables | 138 |
| Target variable | 1 |
| Total modeling columns | 139 |
| Duplicate observations | 0 |

Key preprocessing tasks completed:

- Created and validated the prediction target.
- Engineered receiver efficiency metrics.
- Removed identifiers and administrative metadata.
- Removed observations without a valid prediction target.
- Performed data quality assessment.
- Produced a clean modeling dataset for machine learning.

The dataset is ready for model development and evaluation.